In [8]:
#used to allow importing of physicsnemo
import sys
sys.path.append("/global/u2/k/kfrields/climsim-kaggle-edition")
from physicsnemo.models.diffusion import *
from baseline_models.unet.training_default.climsim_datasets import *
import numpy as np
from sklearn.metrics import r2_score
import torch
import os, gc
from climsim_utils.data_utils import *
from tqdm import tqdm
from matplotlib import cm
import torch
import statsmodels.api as sm # For QQ plot
from baseline_models.unet.training_default.joint_model import JointModel
from omegaconf import OmegaConf
from torch.utils.data import Dataset, DataLoader

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#device = torch.device('cpu')

In [10]:
def apply_temperature_rules(T):
    # Create an output tensor, initialized to zero
    output = np.zeros_like(T)

    # Apply the linear transition within the range 253.16 to 273.16
    mask = (T >= 253.16) & (T <= 273.16)
    output[mask] = (T[mask] - 253.16) / 20.0  # 20.0 is the range (273.16 - 253.16)

    # Values where T > 273.16 set to 1
    output[T > 273.16] = 1

    # Values where T < 253.16 are already set to 0 by the initialization
    return output

def preprocessing_v2_rh_mc(data, input_path, target_path, input_sub, input_div, lbd_qn, out_scale):
    npy_input = np.load(input_path)
    npy_target = np.load(target_path)

    surface_pressure = npy_input[:, data.ps_index]
    
    hyam_component = (data.hyam * data.p0)[np.newaxis,:]
    hybm_component = data.hybm[np.newaxis,:] * surface_pressure[:, np.newaxis]
    
    pressures = hyam_component + hybm_component
    pressures = pressures.reshape(-1,384,60)
    
    pressures_binned = data.zonal_bin_weight_3d(pressures)
    
    actual_input = npy_input.copy().reshape(-1, data.num_latlon, data.input_feature_len)

    npy_input[:,120:180] = 1 - np.exp(-npy_input[:,120:180] * lbd_qn)
    npy_input = (npy_input - input_sub)/input_div
    npy_input = np.where(np.isnan(npy_input), 0, npy_input)
    npy_input = np.where(np.isinf(npy_input), 0, npy_input)
    npy_input[:,120:120+15] = 0
    npy_input[:,60:120] = np.clip(npy_input[:,60:120], 0, 1.2)
    torch_input = torch.tensor(npy_input).float()

    reshaped_target = npy_target.reshape(-1, data.num_latlon, data.target_feature_len)

    t_before = actual_input[:, :, 0:60]
    qn_before = actual_input[:, :, 120:180]
    liq_frac_before = apply_temperature_rules(t_before)
    qc_before = liq_frac_before * qn_before
    qi_before = (1 - liq_frac_before) * qn_before

    t_new = t_before + reshaped_target[:, :, 0:60]*1200
    qn_new = qn_before + reshaped_target[:, :, 120:180]*1200
    liq_frac_new = apply_temperature_rules(t_new)
    qc_new = liq_frac_new * qn_new
    qi_new = (1 - liq_frac_new) * qn_new
    
    actual_target = np.concatenate((reshaped_target[:, :, 0:120], 
                                    (qc_new - qc_before)/1200, 
                                    (qi_new - qi_before)/1200, 
                                    reshaped_target[:, :, 180:240], 
                                    reshaped_target[:, :, 240:]), axis=2)
    return torch_input, actual_input, actual_target, pressures_binned

In [11]:
base_path = '/pscratch/sd/k/kfrields/hugging/E3SM-MMF_saved_models/diffusion_models/refactored_testing_1000_steps_2/'

def load_model(config_path):
    with open(config_path, "r") as f:
        cfg=OmegaConf.load(f)
    base_path = os.path.join(cfg.save_path, cfg.expname)
    v2_rh_mc_input_path =  '/pscratch/sd/j/jerrylin/hugging/E3SM-MMF_ne4/preprocessing/v2_rh_mc/val_set/val_input.npy'
    v2_rh_mc_target_path = '/pscratch/sd/j/jerrylin/hugging/E3SM-MMF_ne4/preprocessing/v2_rh_mc/val_set/val_target.npy'

    #============creates normalization metrics========
    grid_info = xr.open_dataset(cfg.grid_info_path)
    input_mean = xr.open_dataset(cfg.input_mean_path)
    input_max = xr.open_dataset(cfg.input_max_path)
    input_min = xr.open_dataset(cfg.input_min_path)
    output_scale = xr.open_dataset(cfg.output_scale_path)
    qn_lbd = np.loadtxt(cfg.qn_lbd_path, delimiter = ',')
    
    res_std = torch.load(cfg.res_std_path).to(device)
    res_std = res_std.to(torch.float32)
    
    res_mean = torch.load(cfg.res_mean_path).to(device)
    res_mean = res_mean.to(torch.float32)
    
    preds_std = torch.load(cfg.preds_std_path).to(device)
    preds_std = preds_std.to(torch.float32)
    
    preds_mean = torch.load(cfg.preds_mean_path).to(device)
    preds_mean = preds_mean.to(torch.float32)
    
    data = data_utils(grid_info = grid_info, 
                      input_mean = input_mean, 
                      input_max = input_max, 
                      input_min = input_min, 
                      output_scale = output_scale,
                      qinput_log = False,
                      normalize = False,
                      res_std = res_std,
                      res_mean = res_mean,
                      preds_std = preds_std,
                      preds_mean = preds_mean)
    data.set_to_v2_rh_mc_vars()
    
    input_sub_v2_rh_mc, input_div_v2_rh_mc, out_scale_v2_rh_mc = data.save_norm(write=False) # this extracts only the relevant variables
    input_sub_v2_rh_mc = input_sub_v2_rh_mc[None, :]
    input_div_v2_rh_mc = input_div_v2_rh_mc[None, :]
    out_scale_v2_rh_mc = out_scale_v2_rh_mc[None, :]


    deterministic_model_path = os.path.join(base_path, 'unet_model.pt')
    deterministic_model = torch.jit.load(deterministic_model_path).to(device)
    deterministic_model.eval()
    
    #Load residual model
    diff_path = os.path.join(base_path, 'diff_model.pt')
    diff_model = torch.load(diff_path).to(device)
    diff_model.eval()
    
    joint_model = JointModel(deterministic_model,diff_model, res_std, res_mean, 
                                preds_std, preds_mean, 
                                input_profile_num = data.input_profile_num, 
                                input_scalar_num = data.input_scalar_num,
                                target_profile_num = data.target_profile_num,
                                target_scalar_num = data.target_scalar_num, 
                                condition_channel_num = data.target_profile_num  + data.target_scalar_num + data.input_profile_num + data.input_scalar_num,
                                p_mean = cfg.diffusion_model.p_mean,
                                p_std = cfg.diffusion_model.p_std).to(device)

    torch_input, actual_input, actual_target, pressures_binned = preprocessing_v2_rh_mc(data = data, 
                                                                          input_path = v2_rh_mc_input_path, 
                                                                          target_path = v2_rh_mc_target_path, 
                                                                          input_sub = input_sub_v2_rh_mc, 
                                                                          input_div = input_div_v2_rh_mc, 
                                                                          lbd_qn = qn_lbd, 
                                                                          out_scale = out_scale_v2_rh_mc)
    torch_target = torch.tensor(actual_target)
    
    #return joint_model,data, torch_input_v2_rh_mc, actual_input_v2_rh_mc, out_scale_v2_rh_mc, actual_target, pressures_binned, reshaped_target, original_target
    return joint_model, actual_input, torch_input, torch_target, pressures_binned, data, out_scale_v2_rh_mc
    


In [12]:
config_path = '/pscratch/sd/k/kfrields/hugging/E3SM-MMF_saved_models/diffusion_models/refactored_testing/saved_config.yaml'



joint_model, actual_input, torch_input, actual_target, pressures_binned, data, out_scale = load_model(config_path)

/tmp/ipykernel_2207167/4154012725.py:32: RuntimeWarning: divide by zero encountered in divide
  npy_input = (npy_input - input_sub)/input_div


In [2]:
import numpy as np

val_preds = np.load('/global/homes/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/val_preds_epoch_1.npy')
val_targets = np.load('/global/homes/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/val_targets_epoch_1.npy')
val_res = val_targets - val_preds

val_res = val_res.reshape(-1, val_preds.shape[2])
val_preds = val_preds.reshape(-1, val_preds.shape[2])


In [5]:
import torch
preds_std = val_preds.std(axis=0)
torch.save(torch.from_numpy(preds_std), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/remade_preds_std.pt')

preds_mean = val_preds.mean(axis=0)
torch.save(torch.from_numpy(preds_mean), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/remade_preds_mean.pt')


res_std = val_res.std(axis=0)
torch.save(torch.from_numpy(res_std), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/remade_res_std.pt')

res_mean = val_res.mean(axis=0)
torch.save(torch.from_numpy(res_mean), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/remade_res_mean.pt')

In [ ]:
train_preds_load = np.load('/global/homes/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/train_preds_epoch_1.npy')
train_targets = np.load('/global/homes/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/train_targets_epoch_1.npy')
train_res = train_targets - train_preds

train_res = train_res.reshape(-1, train_preds.shape[2])
train_preds = train_preds.reshape(-1, train_preds.shape[2])


In [ ]:
preds_std = train_preds.std(axis=0)
torch.save(torch.from_numpy(preds_std), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/remade_preds_std.pt')

preds_mean = train_preds.mean(axis=0)
torch.save(torch.from_numpy(preds_mean), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/remade_preds_mean.pt')

res_std = train_res.std(axis=0)
torch.save(torch.from_numpy(res_std), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/remade_res_std.pt')

res_mean = train_res.mean(axis=0)
torch.save(torch.from_numpy(res_mean), '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default' + '/remade_res_mean.pt')

In [6]:
#'/preds_mean.pt'

def reshape_tensor(path_end):
    preds_mean = torch.load('/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/' + path_end + '.pt')
    mean_profile = preds_mean[:data.target_profile_num*60]
    mean_scalar = preds_mean[data.target_profile_num*60:]
    
    # reshape x_profile to (batch, input_profile_num, levels)
    mean_profile = mean_profile.reshape(data.target_profile_num, 60)
    # broadcast x_scalar to (batch, input_scalar_num, levels)
    mean_scalar = mean_scalar.unsqueeze(1).expand( -1, 60)
    
    #concatenate x_profile, x_scalar, x_loc to (batch, input_profile_num+input_scalar_num, levels)
    preds_mean = torch.cat((mean_profile, mean_scalar), dim=0)
    
    # pads the beginning of levels so that levels = seq_resolution (which by default is 64)
    preds_mean = torch.nn.functional.pad(preds_mean, (4,0), "constant", 0.0)
    new_path = '/global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/reshaped_'+ path_end + '.pt'
    torch.save(preds_mean, new_path)
    print(f'saved to {new_path}')

In [13]:
reshape_tensor('remade_preds_mean')
reshape_tensor('remade_preds_std')
reshape_tensor('remade_res_std')
reshape_tensor('remade_res_mean')



saved to /global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/reshaped_remade_preds_mean.pt
saved to /global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/reshaped_remade_preds_std.pt
saved to /global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/reshaped_remade_res_std.pt
saved to /global/u2/k/kfrields/climsim-kaggle-edition/baseline_models/unet/training_default/reshaped_remade_res_mean.pt
